In [7]:
import os
import xarray as xr
import numpy as np

In [3]:
# === Processing function ===
def calculate_maximum_6month_mean_prect(start_year, end_year, monthly_mda8, prect):
    years = list(range(start_year, end_year + 1))

    max_vals = []

    for year in years:
        # Define window: Jan of this year to Mar of next year
        start = f"{year}-01"
        end = f"{year + 1}-03"

        # Subset to this window
        subset = monthly_mda8.sel(time=slice(start, end))

        # Compute 6-month rolling mean along time
        rolling_6m = subset.rolling(time=6, center=False).mean()

        # Max 6-month O3 and index
        idx_of_max = rolling_6m.argmax(dim="time")  # (lat, lon)

        # Now build indices for 6-month window
        idx_window = xr.DataArray(
            np.arange(-5, 1),
            dims=["window"],
        ) + idx_of_max.expand_dims(window=6)

        # Mask invalid indices
        valid = (idx_window >= 0) & (idx_window < subset.time.size)
        idx_window = idx_window.where(valid, 0)

        # Use isel smartly
        prect_selected = prect.isel(time=idx_window)  # shape (window, lat, lon)

        # Now average over the 8-hour window
        prect_6m_mean = prect_selected.where(valid).mean(dim="window", skipna=True)

        # Expand dimensions for consistent output
        max_val = prect_6m_mean.expand_dims(year=[year])

        max_vals.append(max_val)

    # Combine across years
    annual_max_6m_no2 = xr.concat(max_vals, dim="year")

    return annual_max_6m_no2

In [6]:
# === Path config ===
O3_DIR = "/glade/work/awells/air_quality/CESM/MDA8/"
PRECT_DIR = "/glade/work/awells/air_quality/CESM/PRECT/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/PRECT_6M/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates_o3 = "20350101-20691231"
            dates_p = "203501-206912"
            new_dates = "2035-2069"
        else:
            dates_o3 = "20200101-20691231"
            dates_p = "202001-206912"
            new_dates = "2020-2069"
        file_list_o3 = [f"{O3_DIR}MDA8_CESM2_{scenario}_{ens_num:02d}_{dates_o3}.nc"]
        file_list_no2 = [f"{PRECT_DIR}PRECT_CESM2_{scenario}_{ens_num:02d}_{dates_p}.nc"]
        PRECT_6m = []

        for f_p, f_o3 in zip(file_list_no2, file_list_o3):
            for f in [f_p, f_o3]:
                if not os.path.exists(f):
                    raise ValueError(f"Missing: {f}")

            print(f"Reading {os.path.basename(f_p)}")
            monthly_mda8 = xr.open_dataarray(f_o3)
            prect = xr.open_dataarray(f_p)

            # Create list of years to calculate over
            start_year = int(str(monthly_mda8.time.dt.year[0].values))
            end_year = int(str(monthly_mda8.time.dt.year[-1].values))  # final year will be 12 months rather than 15

            annual_max_6m_prect = calculate_maximum_6month_mean_prect(start_year, end_year, monthly_mda8, prect)

            PRECT_6m.append(annual_max_6m_prect)

        if PRECT_6m:
            combined = xr.concat(PRECT_6m, dim="year")

            out_file = f"PRECT_6m_CESM2_{scenario}_{ens_num:02d}_{new_dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            combined.to_netcdf(out_path)

print("All processing complete.")

Processing ARISE, Ensemble 01
Reading PRECT_CESM2_ARISE_01_203501-206912.nc
Saving to /glade/work/awells/air_quality/CESM/PRECT_6M/PRECT_6m_CESM2_ARISE_01_203501-206912.nc
Processing ARISE, Ensemble 02
Reading PRECT_CESM2_ARISE_02_203501-206912.nc
Saving to /glade/work/awells/air_quality/CESM/PRECT_6M/PRECT_6m_CESM2_ARISE_02_203501-206912.nc
Processing ARISE, Ensemble 03
Reading PRECT_CESM2_ARISE_03_203501-206912.nc
Saving to /glade/work/awells/air_quality/CESM/PRECT_6M/PRECT_6m_CESM2_ARISE_03_203501-206912.nc
Processing ARISE, Ensemble 04
Reading PRECT_CESM2_ARISE_04_203501-206912.nc
Saving to /glade/work/awells/air_quality/CESM/PRECT_6M/PRECT_6m_CESM2_ARISE_04_203501-206912.nc
Processing ARISE, Ensemble 05
Reading PRECT_CESM2_ARISE_05_203501-206912.nc
Saving to /glade/work/awells/air_quality/CESM/PRECT_6M/PRECT_6m_CESM2_ARISE_05_203501-206912.nc
Processing ARISE, Ensemble 06
Reading PRECT_CESM2_ARISE_06_203501-206912.nc
Saving to /glade/work/awells/air_quality/CESM/PRECT_6M/PRECT_6m_